# FFT Pricer Calibration

Calibrate Heston parameters `(v0, kappa, theta, sigma_v, rho)` directly to own calculated implied volatilities with the Carr-Madan FFT pricer. This notebook reuses the pricing functions from `Carr_Madan_FFT.ipynb`, calls `heston_price()` inside the objective, and records fitted parameters, total squared IV error, and wall-clock time.

In [1]:
from __future__ import annotations

import csv
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm

PARAMETER_COLUMNS = ["v0", "kappa", "theta", "sigma_v", "rho"]

## Load Existing FFT Code

In [2]:
# Reuse the Carr-Madan FFT implementation already developed in this project.
%run Carr_Madan_FFT.ipynb

Black-Scholes collapse check
K=  70.0  FFT= 32.235176  BS= 32.235176  abs_err=2.88e-10
K=  80.0  FFT= 23.223991  BS= 23.223991  abs_err=2.97e-12
K=  90.0  FFT= 15.429227  BS= 15.429227  abs_err=7.13e-11
K= 100.0  FFT=  9.413403  BS=  9.413403  abs_err=7.10e-09
K= 110.0  FFT=  5.293398  BS=  5.293398  abs_err=2.49e-09
K= 120.0  FFT=  2.766558  BS=  2.766558  abs_err=1.74e-09
K= 130.0  FFT=  1.357670  BS=  1.357670  abs_err=3.30e-10
max_abs_err=7.10e-09



Heston FFT vs Monte Carlo check
FFT price       = 9.244531
MC price        = 9.215855
MC 95% interval = [9.148174, 9.283536]
abs_diff        = 0.028676


## Settings

In [3]:
MARKET_DATA_PATH = Path("data/spy_options_clean_with_iv.csv")
TRAINING_DATA_PATH = Path("data/simulated_training_data.csv")
OUTPUT_DIR = Path("outputs/fft_calibration")

RISK_FREE_RATE = 0.0367
MAXITER = 80
FTOL = 1e-9

# Match the FFT settings used in the training-data notebook.
FFT_ALPHA = 1.5
N_FFT = 2**11
FFT_ETA = 0.25

# Use "all", "call", or "put".
OPTION_TYPE = "all"

# Keep quoted IVs positive and inside the range used by the IV solver.
MIN_MARKET_IV = 0.0
MAX_MARKET_IV = 5.0

## Data Helpers

In [4]:
def load_market_data(path: Path) -> pd.DataFrame:
    # Read own calculated IVs and convert expiry dates into year fractions.
    required = {"strike", "expiry", "pull_date", "iv_own", "S0"}
    market = pd.read_csv(path)
    missing = sorted(required - set(market.columns))
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")

    expiry = pd.to_datetime(market["expiry"], errors="coerce")
    pull_date = pd.to_datetime(market["pull_date"], errors="coerce")
    market = market.assign(
        K=pd.to_numeric(market["strike"], errors="coerce"),
        S0=pd.to_numeric(market["S0"], errors="coerce"),
        T=(expiry - pull_date).dt.days / 365.0,
        market_iv=pd.to_numeric(market["iv_own"], errors="coerce"),
    )

    if "dividend_yield" not in market.columns:
        market["dividend_yield"] = 0.0
    market["dividend_yield"] = pd.to_numeric(market["dividend_yield"], errors="coerce").fillna(0.0)

    finite = np.isfinite(market[["K", "S0", "T", "market_iv", "dividend_yield"]]).all(axis=1)
    positive = (market["S0"] > 0.0) & (market["T"] > 0.0)
    iv_range = market["market_iv"].between(MIN_MARKET_IV, MAX_MARKET_IV)
    return market.loc[finite & positive & iv_range].copy()


def parameter_bounds(training_data: Path) -> list[tuple[float, float]]:
    # Use the same parameter range that generated the synthetic Heston surfaces.
    training = pd.read_csv(training_data, usecols=PARAMETER_COLUMNS)
    return [
        (float(training[column].min()), float(training[column].max()))
        for column in PARAMETER_COLUMNS
    ]


def initial_guess(bounds: list[tuple[float, float]]) -> np.ndarray:
    return np.array([(low + high) / 2.0 for low, high in bounds], dtype=float)


def make_contract_groups(market: pd.DataFrame) -> list[dict[str, object]]:
    # Precompute maturity groups so the objective does not rebuild them every call.
    groups = []
    for _, group in market.groupby(["S0", "T", "dividend_yield"], sort=False):
        groups.append({
            "positions": group.index.to_numpy(),
            "strikes": group["K"].to_numpy(dtype=float),
            "market_iv": group["market_iv"].to_numpy(dtype=float),
            "S0": float(group["S0"].iloc[0]),
            "T": float(group["T"].iloc[0]),
            "q": float(group["dividend_yield"].iloc[0]),
        })
    return groups

## Pricing and IV Helpers

In [5]:
def params_to_dict(values: np.ndarray) -> dict[str, float]:
    return {column: float(value) for column, value in zip(PARAMETER_COLUMNS, values)}


def heston_price(strikes, S0, T, r, q, params):
    # Carr_Madan_FFT.ipynb names vol-of-vol "sigma"; calibration uses "sigma_v".
    fft_params = {
        "v0": params["v0"],
        "kappa": params["kappa"],
        "theta": params["theta"],
        "sigma": params["sigma_v"],
        "rho": params["rho"],
    }
    return carr_madan_fft_call(
        strikes,
        S0=S0,
        T=T,
        r=r,
        q=q,
        params=fft_params,
        alpha=FFT_ALPHA,
        N=N_FFT,
        eta=FFT_ETA,
    )


def implied_volatility_from_prices(prices, S0, strikes, T, r, q, initial_iv):
    # Newton solve all IVs in a group at once, starting from market IVs.
    prices = np.asarray(prices, dtype=float)
    strikes = np.asarray(strikes, dtype=float)
    sigma = np.clip(np.asarray(initial_iv, dtype=float), 1e-4, 5.0)

    intrinsic = np.maximum(S0 * np.exp(-q * T) - strikes * np.exp(-r * T), 0.0)
    upper_bound = S0 * np.exp(-q * T)
    epsilon = 1e-8
    prices = np.nan_to_num(prices, nan=intrinsic + epsilon, posinf=upper_bound - epsilon)
    prices = np.clip(prices, intrinsic + epsilon, upper_bound - epsilon)

    for _ in range(12):
        vol_sqrt_t = sigma * np.sqrt(T)
        d1 = (np.log(S0 / strikes) + (r - q + 0.5 * sigma**2) * T) / vol_sqrt_t
        d2 = d1 - vol_sqrt_t
        bs_price = S0 * np.exp(-q * T) * norm.cdf(d1) - strikes * np.exp(-r * T) * norm.cdf(d2)
        vega = S0 * np.exp(-q * T) * norm.pdf(d1) * np.sqrt(T)
        step = np.where(vega > 1e-10, (bs_price - prices) / vega, 0.0)
        sigma = np.clip(sigma - step, 1e-4, 5.0)

    return sigma

## Calibration Helpers

In [6]:
def predict_market_iv(params: dict[str, float], groups: list[dict[str, object]], n_contracts: int) -> np.ndarray:
    # Price each maturity group once, then convert prices to model IVs.
    predicted_iv = np.full(n_contracts, np.nan, dtype=float)

    for group in groups:
        prices = heston_price(
            group["strikes"],
            group["S0"],
            group["T"],
            RISK_FREE_RATE,
            group["q"],
            params,
        )
        predicted_iv[group["positions"]] = implied_volatility_from_prices(
            prices,
            group["S0"],
            group["strikes"],
            group["T"],
            RISK_FREE_RATE,
            group["q"],
            group["market_iv"],
        )

    return predicted_iv


def squared_iv_error(predicted_iv: np.ndarray, market_iv: np.ndarray) -> float:
    # Penalize invalid IVs so the optimizer moves away from unstable guesses.
    valid = np.isfinite(predicted_iv)
    residual = predicted_iv[valid] - market_iv[valid]
    invalid_penalty = 100.0 * np.sum(~valid)
    return float(np.sum(residual * residual) + invalid_penalty)

## Run Calibration

In [7]:
# Load and optionally filter market contracts.
market = load_market_data(MARKET_DATA_PATH)
if OPTION_TYPE != "all" and "type" in market.columns:
    market = market.loc[market["type"].str.lower() == OPTION_TYPE].copy()

if market.empty:
    raise ValueError("No market contracts remain after filtering.")

market = market.reset_index(drop=True)
market_iv = market["market_iv"].to_numpy(dtype=float)
contract_groups = make_contract_groups(market)
bounds = parameter_bounds(TRAINING_DATA_PATH)


def objective(values: np.ndarray) -> float:
    # Price with heston_price(), convert to IV, then sum squared IV errors.
    predicted_iv = predict_market_iv(params_to_dict(values), contract_groups, len(market))
    return squared_iv_error(predicted_iv, market_iv)


# L-BFGS-B is the same bounded optimizer style used for the GP calibration.
started_at = time.perf_counter()
result = minimize(
    objective,
    x0=initial_guess(bounds),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": MAXITER, "ftol": FTOL},
)
wall_clock_seconds = time.perf_counter() - started_at

# Reprice once at the fitted parameters for reporting.
fitted_params = np.asarray(result.x, dtype=float)
predicted_iv = predict_market_iv(params_to_dict(fitted_params), contract_groups, len(market))
residual = predicted_iv - market_iv
squared_error = residual * residual
valid_iv = np.isfinite(predicted_iv)
total_error = float(np.nansum(squared_error))

summary = {
    "success": bool(result.success),
    "message": str(result.message),
    "contracts": int(len(market)),
    "valid_model_ivs": int(valid_iv.sum()),
    "evaluations": int(result.nfev),
    "fitted_parameters": params_to_dict(fitted_params),
    "total_calibration_error": total_error,
    "rmse": float(np.sqrt(np.nanmean(squared_error))),
    "mae": float(np.nanmean(np.abs(residual))),
    "wall_clock_seconds": float(wall_clock_seconds),
    "bounds": {column: list(bound) for column, bound in zip(PARAMETER_COLUMNS, bounds)},
    "market_data": str(MARKET_DATA_PATH),
    "risk_free_rate": RISK_FREE_RATE,
    "option_type": OPTION_TYPE,
    "fft_settings": {"alpha": FFT_ALPHA, "n_fft": N_FFT, "eta": FFT_ETA},
}

summary

{'success': True,
 'message': 'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH',
 'contracts': 899,
 'valid_model_ivs': 899,
 'evaluations': 492,
 'fitted_parameters': {'v0': 0.021829300825648913,
  'kappa': 2.3550045638746653,
  'theta': 0.0101229951136501,
  'sigma_v': 0.9993863554271026,
  'rho': -0.9495768662885722},
 'total_calibration_error': 5.738206319286743,
 'rmse': 0.07989290885075385,
 'mae': 0.04525350942625031,
 'wall_clock_seconds': 3.4745114580146037,
 'bounds': {'v0': [0.0100207958590628, 0.2498920421524241],
  'kappa': [0.1001058627817415, 4.998386022881946],
  'theta': [0.0101229951136501, 0.2499243545643585],
  'sigma_v': [0.0501999396475361, 0.9993863554271026],
  'rho': [-0.9495768662885722, -0.0005757410491344]},
 'market_data': 'data/spy_options_clean_with_iv.csv',
 'risk_free_rate': 0.0367,
 'option_type': 'all',
 'fft_settings': {'alpha': 1.5, 'n_fft': 2048, 'eta': 0.25}}

## Save Results

In [8]:
# Save contract-level model IVs and residuals.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predictions = market.copy()
for column, value in zip(PARAMETER_COLUMNS, fitted_params):
    predictions[f"fitted_{column}"] = value

predictions["fft_predicted_iv"] = predicted_iv
predictions["iv_residual"] = residual
predictions["iv_squared_error"] = squared_error
predictions.to_csv(OUTPUT_DIR / "calibration_predictions.csv", index=False)

# Save the fitted parameters and error metrics.
with (OUTPUT_DIR / "calibration_summary.json").open("w") as f:
    json.dump(summary, f, indent=2)
    f.write("\n")

with (OUTPUT_DIR / "calibration_summary.csv").open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        *PARAMETER_COLUMNS,
        "total_calibration_error",
        "rmse",
        "mae",
        "wall_clock_seconds",
        "contracts",
        "valid_model_ivs",
        "evaluations",
        "success",
        "message",
    ])
    writer.writerow([
        *[summary["fitted_parameters"][column] for column in PARAMETER_COLUMNS],
        summary["total_calibration_error"],
        summary["rmse"],
        summary["mae"],
        summary["wall_clock_seconds"],
        summary["contracts"],
        summary["valid_model_ivs"],
        summary["evaluations"],
        summary["success"],
        summary["message"],
    ])

print(f"Saved calibration summary to {OUTPUT_DIR / 'calibration_summary.csv'}")
print(f"Saved per-contract predictions to {OUTPUT_DIR / 'calibration_predictions.csv'}")

Saved calibration summary to outputs/fft_calibration/calibration_summary.csv
Saved per-contract predictions to outputs/fft_calibration/calibration_predictions.csv
